# SYNTHIA -> Cityscapes Final Resource Pipeline (500 Images)

This notebook is the final low-resource run:

| Step | Description |
|------|-------------|
| 0 | Mount Drive, clone repo, install deps, set paths |
| 1 | Prepare SYNTHIA and Cityscapes data |
| 2 | Build multilabel JSONs for DAMP |
| 3 | Train/load the DAMP full checkpoint |
| 4 | Generate zero-shot, DAMP prompt-only, and DAMP full CAMs for 500 SYNTHIA images |
| 5 | Build a class-wise hybrid CAM set from zero-shot + DAMP prompt-only |
| 6 | Evaluate zero-shot, prompt-only, DAMP full, and hybrid on the same 500-image split |
| 7 | Generate pseudo masks from the selected CAM kind + selected post-processing method |
| 8 | Export image/mask pairs for segmentation training |

Default best source is `hybrid`. Cell 7 prints zero-shot as report-only and auto-selects among DAMP prompt-only, DAMP full, and hybrid.

In [ ]:
# ===== CELL 0a: MOUNT DRIVE =====
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ===== CELL 0b: CLONE / UPDATE REPO + INSTALL DEPS =====
import os, sys, subprocess

REPO_DIR = '/content/Damp_es'
REPO_URL = 'https://github.com/baominh5xx2/Damp_es_CS338.git'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Repo already exists at {REPO_DIR}; updating with git pull --ff-only')
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=False)
    pull = subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=False)
    if pull.returncode != 0:
        print('WARNING: git pull failed, likely because the Colab repo has local edits.')
        print('If you need a clean update, run: !rm -rf /content/Damp_es and rerun this cell.')

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, check=False)

# Install dependencies.
!pip install -q timm yacs ftfy regex lxml ttach
!pip install -q opencv-python-headless scikit-learn matplotlib tqdm
!pip install -q datasets huggingface_hub pyarrow
!pip install -q git+https://github.com/KaiyangZhou/Dassl.pytorch.git



In [ ]:
# ===== CELL 0c: CONFIG - paths & settings =====
from pathlib import Path

# Change this to your Drive path if needed.
DATA_ROOT = Path('/content/drive/MyDrive/datasets/synthia_cs338')
OUTPUT_DIR = DATA_ROOT / 'output'

# Final low-resource run.
RUN_NAME = 'synthia_clipnorm_tau052_e3'
CAM_MAX_IMAGES = 500
EVAL_MAX_IMAGES = CAM_MAX_IMAGES
GRID_SEARCH_MAX_IMAGES = 100  # legacy knob; Cell 7 scores fixed protocols on full EVAL_MAX_IMAGES
BEST_CAM_KIND = 'hybrid'  # options: 'zero', 'prompt_only', 'damp_full', 'hybrid'
CRF_CONFIDENCE = 0.95
CRF_N_JOBS = 1
USE_CRF = False  # final-resource default: False is much faster
PSEUDO_MASK_THRESHOLD = 0.01  # fallback only; Cell 7 selects the final threshold/method

# Report-only strict zero-shot baseline: no grid-search/postprocess tuning.
ZERO_STRICT_ENABLED = True
ZERO_STRICT_METHOD = 'baseline'  # baseline, norm, boost, no_bg
ZERO_STRICT_THRES = 0.03
ZERO_STRICT_ALPHA = 1.0

# Derived paths.
SYNTHIA_RAW = DATA_ROOT / 'data' / 'raw' / 'synthia'
CITY_RAW = DATA_ROOT / 'data' / 'raw' / 'cityscapes'
PROCESSED = DATA_ROOT / 'data' / 'processed'

DAMP_DIR = OUTPUT_DIR / 'damp' / RUN_NAME
PROMPT_CKPT = DAMP_DIR / 'prompt_learner.pth'
CAM_ZERO_DIR = OUTPUT_DIR / 'synthia' / f'cams_zero_raw_{CAM_MAX_IMAGES}'
CAM_PROMPT_DIR = OUTPUT_DIR / 'synthia' / f'cams_damp_{RUN_NAME}_prompt_only_raw_{CAM_MAX_IMAGES}'
CAM_FULL_DIR = OUTPUT_DIR / 'synthia' / f'cams_damp_{RUN_NAME}_full_raw_{CAM_MAX_IMAGES}'
CAM_HYBRID_DIR = OUTPUT_DIR / 'synthia' / f'cams_hybrid_zero_prompt_{RUN_NAME}_{CAM_MAX_IMAGES}'
CAM_DIR_BY_KIND = {
    'zero': CAM_ZERO_DIR,
    'prompt_only': CAM_PROMPT_DIR,
    'damp_full': CAM_FULL_DIR,
    'hybrid': CAM_HYBRID_DIR,
}
SELECTABLE_CAM_KINDS = ('prompt_only', 'damp_full', 'hybrid')  # zero is report-only; pick from DAMP-family CAMs
BEST_CAM_DIR = CAM_DIR_BY_KIND[BEST_CAM_KIND]

SEG_EXPORT_DIR = OUTPUT_DIR / 'segmentation' / RUN_NAME
MASK_DIR = None  # set in Cell 8 after Cell 7 selects method/threshold
SEG_TRAIN_PAIRS = None  # set in Cell 8

SYNTHIA_IMG = SYNTHIA_RAW / 'images'
SYNTHIA_LBL = SYNTHIA_RAW / 'labels'
SYNTHIA_SPLIT = SYNTHIA_RAW / 'splits' / 'train.txt'
SYNTHIA_CAM_SPLIT = SYNTHIA_RAW / 'splits' / f'train_first{CAM_MAX_IMAGES}.txt'

CITY_IMG = CITY_RAW / 'images'
CITY_LBL = CITY_RAW / 'labels'
CITY_TRAIN_SPLIT = CITY_RAW / 'splits' / 'train.txt'
CITY_VAL_SPLIT = CITY_RAW / 'splits' / 'val.txt'

HF_SYNTHIA_REPO = 'Minhbao5xx2/synthia-rand-cityscapes-16class-parquet_fix'
HF_CITY_REPO = 'Chris1/cityscapes'

print(f'DATA_ROOT        : {DATA_ROOT}')
print(f'RUN_NAME         : {RUN_NAME}')
print(f'CAM_MAX_IMAGES   : {CAM_MAX_IMAGES}')
print(f'EVAL_MAX_IMAGES  : {EVAL_MAX_IMAGES}')
print(f'GRID_SEARCH_MAX  : {GRID_SEARCH_MAX_IMAGES}')
print(f'DAMP_DIR         : {DAMP_DIR}')
print(f'PROMPT_CKPT      : {PROMPT_CKPT}')
print(f'CAM_ZERO_DIR     : {CAM_ZERO_DIR}')
print(f'CAM_PROMPT_DIR   : {CAM_PROMPT_DIR}')
print(f'CAM_FULL_DIR     : {CAM_FULL_DIR}')
print(f'CAM_HYBRID_DIR   : {CAM_HYBRID_DIR}')
print(f'BEST_CAM_KIND    : {BEST_CAM_KIND}')
print(f'BEST_CAM_DIR     : {BEST_CAM_DIR}')
print(f'ZERO_STRICT      : {ZERO_STRICT_ENABLED}, {ZERO_STRICT_METHOD}, t={ZERO_STRICT_THRES}, a={ZERO_STRICT_ALPHA}')
print(f'MASK_DIR         : {MASK_DIR}')
print(f'SEG_TRAIN_PAIRS  : {SEG_TRAIN_PAIRS}')

## Step 1: Download & Prepare SYNTHIA

Downloads from HuggingFace (fixed parquet with correct 16-bit labels), then converts to images/labels/splits.

In [ ]:
# ===== CELL 1: DOWNLOAD + PREPARE SYNTHIA =====
import os, glob
from pathlib import Path

PARQUET_DIR = DATA_ROOT / "synthia_parquet"

# ── 1a: Download parquet from HuggingFace ──
if not SYNTHIA_SPLIT.exists():
    print("Downloading SYNTHIA parquet from HuggingFace ...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=HF_SYNTHIA_REPO,
        repo_type="dataset",
        local_dir=str(PARQUET_DIR),
        max_workers=16,
    )
    print("Download complete.")
else:
    print("SYNTHIA data already prepared, skipping download.")

# ── 1b: Convert parquet → images/labels/splits ──
if not SYNTHIA_SPLIT.exists():
    print("Converting parquet to images/labels/splits ...")
    !python {REPO_DIR}/tools/prepare_synthia_hf.py \
        --parquet-dir {PARQUET_DIR / "parquet"} \
        --output-root {SYNTHIA_RAW} \
        --num-workers 16
else:
    print("SYNTHIA splits already exist, skipping conversion.")

# ── 1c: Verify ──
n_img = len(list(Path(SYNTHIA_IMG).glob("*.png"))) if SYNTHIA_IMG.exists() else 0
n_lbl = len(list(Path(SYNTHIA_LBL).glob("*.png"))) if SYNTHIA_LBL.exists() else 0
n_split = len(open(SYNTHIA_SPLIT).readlines()) if SYNTHIA_SPLIT.exists() else 0
print(f"\nSYNTHIA: {n_img} images, {n_lbl} labels, {n_split} split entries")

# Quick label sanity check
if n_lbl > 0:
    import cv2, numpy as np
    sample_lbl = sorted(Path(SYNTHIA_LBL).glob("*.png"))[0]
    arr = cv2.imread(str(sample_lbl), cv2.IMREAD_UNCHANGED)
    if arr.ndim == 3:
        unique = np.unique(arr[:,:,2])  # Red channel = class ID (16-bit)
    else:
        unique = np.unique(arr)
    n_classes = len([v for v in unique if v != 0])
    print(f"Label check ({sample_lbl.name}): {n_classes} valid classes, unique IDs: {unique.tolist()[:15]}")

## Step 2: Download & Prepare Cityscapes

In [ ]:
# ===== CELL 2: DOWNLOAD + PREPARE CITYSCAPES =====
if not CITY_VAL_SPLIT.exists():
    print("Downloading & preparing Cityscapes from HuggingFace ...")
    !python {REPO_DIR}/tools/prepare_cityscapes_hf.py \
        --dataset-id {HF_CITY_REPO} \
        --output-root {CITY_RAW} \
        --splits train,validation \
        --num-workers 16
else:
    print("Cityscapes already prepared.")

# Verify
n_city_img = len(list(Path(CITY_IMG).glob("*.png"))) if CITY_IMG.exists() else 0
n_city_train = len(open(CITY_TRAIN_SPLIT).readlines()) if CITY_TRAIN_SPLIT.exists() else 0
n_city_val = len(open(CITY_VAL_SPLIT).readlines()) if CITY_VAL_SPLIT.exists() else 0
print(f"Cityscapes: {n_city_img} images, {n_city_train} train, {n_city_val} val")

## Step 3: Build Multilabel JSON

Extracts per-image class labels from segmentation masks for multi-label classification.

In [ ]:
import os

# Patch dassl library to fix numpy compatibility error
dassl_file = '/usr/local/lib/python3.12/dist-packages/dassl/data/transforms/randaugment.py'
if os.path.exists(dassl_file):
    with open(dassl_file, 'r') as f:
        content = f.read()

    # Replace np.int with int
    new_content = content.replace('astype(np.int)', 'astype(int)')

    if content != new_content:
        with open(dassl_file, 'w') as f:
            f.write(new_content)
        print(f"Successfully patched {dassl_file}")
    else:
        print("File already patched or np.int not found.")
else:
    print(f"Could not find {dassl_file}. Please check installation path.")

In [ ]:
# ===== CELL 3: BUILD MULTILABEL JSON =====
SYNTHIA_ML_DIR = PROCESSED / "synthia_multilabel"
CITY_ML_DIR    = PROCESSED / "cityscapes_multilabel"

# ── SYNTHIA multilabel ──
synthia_ml_file = SYNTHIA_ML_DIR / "multilabel.json"
if not synthia_ml_file.exists():
    print("Building SYNTHIA multilabel ...")
    !python {REPO_DIR}/tools/build_synthia_multilabel.py \
        --split-file {SYNTHIA_SPLIT} \
        --label-dir {SYNTHIA_LBL} \
        --output-dir {SYNTHIA_ML_DIR} \
        --num-workers 16
else:
    print("SYNTHIA multilabel already exists.")

# ── Cityscapes train multilabel ──
city_train_ml = CITY_ML_DIR / "train_multilabel.json"
if not city_train_ml.exists():
    print("Building Cityscapes train multilabel ...")
    !python {REPO_DIR}/tools/build_cityscapes_multilabel.py \
        --split-file {CITY_TRAIN_SPLIT} \
        --label-dir {CITY_LBL} \
        --output-dir {CITY_ML_DIR} \
        --output-file train_multilabel.json \
        --num-workers 16
else:
    print("Cityscapes train multilabel already exists.")

# ── Cityscapes val multilabel ──
city_val_ml = CITY_ML_DIR / "val_multilabel.json"
if not city_val_ml.exists():
    print("Building Cityscapes val multilabel ...")
    !python {REPO_DIR}/tools/build_cityscapes_multilabel.py \
        --split-file {CITY_VAL_SPLIT} \
        --label-dir {CITY_LBL} \
        --output-dir {CITY_ML_DIR} \
        --output-file val_multilabel.json \
        --num-workers 16
else:
    print("Cityscapes val multilabel already exists.")

print("\nAll multilabel files ready!")

## Step 4: Train or Load DAMP Full Checkpoint

`configs/trainers/damp_synthia_fast.yaml` is now the source of truth for the debugged run:

- 3 epochs for `prompt_learner` and `context_decoder`
- CLIP pixel normalization, not ImageNet normalization
- `TRAINER.DAMP.TAU = 0.52`
- `TRAINER.DAMP.PSEUDO_TEMP = 0.0`, meaning logits are calibrated by CLIP logit scale before sigmoid
- checkpoint every epoch

The generated `prompt_learner.pth` includes both `prompt_learner` and `context_decoder`, so CAM generation without `--damp_disable_decoder` is DAMP full.


In [ ]:
# ===== CELL 4: TRAIN / LOAD DAMP FULL =====
%cd {REPO_DIR}

if PROMPT_CKPT.exists():
    print(f'DAMP checkpoint already exists: {PROMPT_CKPT}')
    print('Delete the run directory if you want to retrain from scratch.')
else:
    print(f'Training DAMP full checkpoint -> {DAMP_DIR}')
    !python train.py \
        --config-file configs/trainers/damp_synthia_fast.yaml \
        DATASET.ROOT {DATA_ROOT} \
        OUTPUT_DIR {DAMP_DIR}

# Verify.
if PROMPT_CKPT.exists():
    import os
    size_mb = os.path.getsize(PROMPT_CKPT) / 1024 / 1024
    print(f'\nPrompt checkpoint: {PROMPT_CKPT} ({size_mb:.1f} MB)')
else:
    raise FileNotFoundError(f'prompt_learner.pth not found: {PROMPT_CKPT}')


## Step 5: Generate Zero-shot, DAMP Prompt-only, and DAMP Full CAMs for 500 Images

This cell generates the CAM sets used in the final comparison:

- `zero`: CLIP-ES zero-shot CAMs.
- `prompt_only`: learned DAMP prompt only, with `context_decoder` disabled. This is an ablation, not the full method.
- `damp_full`: learned DAMP prompt plus `context_decoder`.

CAM generation uses high resolution and attention refinement. Existing complete CAM folders are skipped. Delete a CAM folder if you need to regenerate it with the current settings.

In [ ]:
# ===== CELL 5: GENERATE ZERO-SHOT + DAMP PROMPT-ONLY + DAMP FULL CAMs (500 images) =====
%cd {REPO_DIR}

import glob
import numpy as np
import subprocess
from pathlib import Path
from PIL import Image

PROMPT_CKPT = DAMP_DIR / 'prompt_learner.pth'
if not PROMPT_CKPT.exists():
    raise FileNotFoundError(f'DAMP checkpoint not found: {PROMPT_CKPT}. Run Cell 4 first.')

# Create an explicit split so CAMs, metrics, masks, and train pairs match exactly.
SYNTHIA_CAM_SPLIT.parent.mkdir(parents=True, exist_ok=True)
with open(SYNTHIA_SPLIT, 'r') as f:
    entries = [line.strip() for line in f if line.strip()]
entries_subset = entries[:CAM_MAX_IMAGES]
with open(SYNTHIA_CAM_SPLIT, 'w', newline='\n') as f:
    for entry in entries_subset:
        f.write(str(Path(entry).name) + '\n')
with open(SYNTHIA_CAM_SPLIT, 'r') as f:
    cam_split_entries = [line.strip() for line in f if line.strip()]
print(f'Wrote CAM split: {SYNTHIA_CAM_SPLIT} ({len(cam_split_entries)} entries)')
print(f'First CAM split entries: {cam_split_entries[:3]}')
if len(cam_split_entries) != len(entries_subset):
    raise RuntimeError(f'CAM split write/read mismatch: wrote {len(entries_subset)}, read {len(cam_split_entries)}')
if not cam_split_entries:
    raise RuntimeError('CAM split is empty.')
sample_entry = cam_split_entries[0]
sample_img = SYNTHIA_IMG / Path(sample_entry).name
sample_lbl = SYNTHIA_LBL / Path(sample_entry).name
print(f'Sample paths exist: image={sample_img.exists()} label={sample_lbl.exists()} entry={sample_entry}')
if not sample_img.exists() or not sample_lbl.exists():
    raise FileNotFoundError(f'Sample image/label missing for CAM split entry: {sample_entry}')
sample_ids = sorted(int(v) for v in np.unique(np.asarray(Image.open(sample_lbl))))
sample_valid_ids = [v for v in sample_ids if v in {0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 15, 17, 18}]
print(f'Sample label valid ids: {sample_valid_ids[:20]}')
if not sample_valid_ids:
    raise RuntimeError(f'Sample label has no SYNTHIA/Cityscapes valid ids: {sample_lbl}')

for d in (CAM_ZERO_DIR, CAM_PROMPT_DIR, CAM_FULL_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Change these if you need a cheaper rerun.
MAX_LONG_SIDE = 2048
USE_REFINE = True
# One GPU worker is safer on Colab T4. Multiple workers load multiple CLIP/DAMP copies on the same GPU.
NUM_CAM_WORKERS = 1
GENERATE_DAMP_FULL = True

def _arg_path(args, flag):
    idx = args.index(flag)
    return Path(args[idx + 1])

def run_generate(args):
    if not USE_REFINE:
        args.append('--no_refine')
    cmd = [str(x) for x in args]
    out_dir = _arg_path(args, '--cam_out_dir')
    print(' '.join(cmd))
    proc = subprocess.run(cmd, cwd=str(REPO_DIR), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log = proc.stdout or ''
    if len(log) > 12000:
        print('--- generator log tail ---')
        print(log[-12000:])
    else:
        print(log)
    if proc.returncode != 0:
        raise RuntimeError(f'CAM generation command failed with return code {proc.returncode}')
    n_out = len(glob.glob(str(out_dir / '*.npy')))
    if n_out == 0:
        raise RuntimeError(f'CAM generation finished but wrote 0 files to {out_dir}. See generator log above.')

# 1. Zero-shot CLIP-ES baseline.
n_zero = len(glob.glob(str(CAM_ZERO_DIR / '*.npy')))
if n_zero >= CAM_MAX_IMAGES:
    print(f'Zero-shot CAMs already exist ({n_zero}).')
else:
    print(f'Generating zero-shot CAMs -> {CAM_ZERO_DIR} (max_long_side={MAX_LONG_SIDE})')
    run_generate([
        'python', 'generate_cams.py',
        '--dataset', 'synthia',
        '--img_root', SYNTHIA_IMG,
        '--label_root', SYNTHIA_LBL,
        '--split_file', SYNTHIA_CAM_SPLIT,
        '--cam_out_dir', CAM_ZERO_DIR,
        '--cam_score', 'softmax',
        '--max_images', CAM_MAX_IMAGES,
        '--max_long_side', MAX_LONG_SIDE,
        '--num_workers', NUM_CAM_WORKERS,
        '--skip_existing',
    ])

# 2. DAMP prompt-only ablation: disables context_decoder.
n_prompt = len(glob.glob(str(CAM_PROMPT_DIR / '*.npy')))
if n_prompt >= CAM_MAX_IMAGES:
    print(f'DAMP prompt-only CAMs already exist ({n_prompt}).')
else:
    print(f'Generating DAMP prompt-only CAMs -> {CAM_PROMPT_DIR} (max_long_side={MAX_LONG_SIDE})')
    run_generate([
        'python', 'generate_cams.py',
        '--dataset', 'synthia',
        '--img_root', SYNTHIA_IMG,
        '--label_root', SYNTHIA_LBL,
        '--split_file', SYNTHIA_CAM_SPLIT,
        '--cam_out_dir', CAM_PROMPT_DIR,
        '--damp_prompt_ckpt', PROMPT_CKPT,
        '--damp_name_mode', 'train',
        '--damp_disable_decoder',
        '--cam_score', 'raw',
        '--max_images', CAM_MAX_IMAGES,
        '--max_long_side', MAX_LONG_SIDE,
        '--num_workers', NUM_CAM_WORKERS,
        '--skip_existing',
    ])

# 3. DAMP full: prompt learner + context_decoder.
n_full = len(glob.glob(str(CAM_FULL_DIR / '*.npy')))
if not GENERATE_DAMP_FULL:
    raise RuntimeError('DAMP full generation is disabled, but this final run requires full DAMP CAMs.')
elif n_full >= CAM_MAX_IMAGES:
    print(f'DAMP full CAMs already exist ({n_full}).')
else:
    print(f'Generating DAMP full CAMs -> {CAM_FULL_DIR} (max_long_side={MAX_LONG_SIDE})')
    run_generate([
        'python', 'generate_cams.py',
        '--dataset', 'synthia',
        '--img_root', SYNTHIA_IMG,
        '--label_root', SYNTHIA_LBL,
        '--split_file', SYNTHIA_CAM_SPLIT,
        '--cam_out_dir', CAM_FULL_DIR,
        '--damp_prompt_ckpt', PROMPT_CKPT,
        '--damp_name_mode', 'train',
        '--cam_score', 'raw',
        '--max_images', CAM_MAX_IMAGES,
        '--max_long_side', MAX_LONG_SIDE,
        '--num_workers', NUM_CAM_WORKERS,
        '--skip_existing',
    ])

n_zero = len(glob.glob(str(CAM_ZERO_DIR / '*.npy')))
n_prompt = len(glob.glob(str(CAM_PROMPT_DIR / '*.npy')))
n_full = len(glob.glob(str(CAM_FULL_DIR / '*.npy')))
print(f'
zero CAMs       : {n_zero} in {CAM_ZERO_DIR}')
print(f'prompt-only CAMs: {n_prompt} in {CAM_PROMPT_DIR}')
print(f'DAMP full CAMs  : {n_full} in {CAM_FULL_DIR}')
if n_zero == 0 or n_prompt == 0 or n_full == 0:
    raise RuntimeError('Missing CAM files. Check generation logs above.')

## Step 6: Build Class-wise Hybrid CAMs

Hybrid is the last low-resource trick: use the source that was stronger by class.

- zero-shot: `road`, `building`, `wall`, `pole`, `car`, `bicycle`
- DAMP prompt-only: `sidewalk`, `fence`, `vegetation`, `person`, `rider`, `bus`
- other classes fall back to whichever source has the class.

This hybrid is an extra `zero + DAMP prompt-only` trick. DAMP full is still generated, evaluated, and selectable as its own full-method CAM kind.

In [ ]:
# ===== CELL 6: MERGE ZERO-SHOT + DAMP PROMPT-ONLY INTO HYBRID CAMs =====
import glob
import numpy as np
from pathlib import Path
from tqdm import tqdm

CAM_TYPE = 'attn_highres'
CAM_HYBRID_DIR.mkdir(parents=True, exist_ok=True)

# Cityscapes train IDs: 0 road, 1 sidewalk, 2 building, 3 wall, 4 fence,
# 5 pole, 8 vegetation, 11 person, 12 rider, 13 car, 15 bus, 18 bicycle.
DAMP_PROMPT_CLASSES = {1, 4, 8, 11, 12, 15}
ZERO_CLASSES = {0, 2, 3, 5, 13, 18}

with open(SYNTHIA_CAM_SPLIT, 'r') as f:
    entries = [line.strip() for line in f if line.strip()]

saved = 0
missing = []
for entry in tqdm(entries, desc='hybrid zero+damp_prompt'):
    stem = Path(entry).stem
    out_path = CAM_HYBRID_DIR / f'{stem}.npy'
    if out_path.exists():
        saved += 1
        continue

    zero_path = CAM_ZERO_DIR / f'{stem}.npy'
    damp_path = CAM_PROMPT_DIR / f'{stem}.npy'
    if not zero_path.exists() or not damp_path.exists():
        missing.append(stem)
        continue

    zero = np.load(zero_path, allow_pickle=True).item()
    damp = np.load(damp_path, allow_pickle=True).item()
    zero_keys = [int(x) for x in zero['keys'].tolist()]
    damp_keys = [int(x) for x in damp['keys'].tolist()]
    zero_map = {k: i for i, k in enumerate(zero_keys)}
    damp_map = {k: i for i, k in enumerate(damp_keys)}
    keys = sorted(set(zero_keys) | set(damp_keys))

    merged = []
    for k in keys:
        use_damp = k in DAMP_PROMPT_CLASSES
        if use_damp and k in damp_map:
            merged.append(damp[CAM_TYPE][damp_map[k]])
        elif (not use_damp) and k in zero_map:
            merged.append(zero[CAM_TYPE][zero_map[k]])
        elif k in damp_map:
            merged.append(damp[CAM_TYPE][damp_map[k]])
        elif k in zero_map:
            merged.append(zero[CAM_TYPE][zero_map[k]])

    if not merged:
        missing.append(stem)
        continue

    out = dict(damp)
    out[CAM_TYPE] = np.stack(merged, axis=0).astype(damp[CAM_TYPE].dtype, copy=False)
    out['keys'] = np.asarray(keys, dtype=np.int64)
    np.save(out_path, out)
    saved += 1

n_hybrid = len(glob.glob(str(CAM_HYBRID_DIR / '*.npy')))
print(f'Hybrid CAMs: {n_hybrid} in {CAM_HYBRID_DIR}')
if missing:
    print(f'WARNING: missing {len(missing)} hybrid entries. First 10: {missing[:10]}')
if n_hybrid == 0:
    raise RuntimeError('No hybrid CAMs generated.')

## Step 7: Simple Parallel CAM Evaluation and Pseudo-mask Param Selection

This is the fast but still straightforward full-resolution version:

- no downsampling
- no max/argmax cache rewrite
- Cell 7 scores a small fixed protocol set directly on full `EVAL_MAX_IMAGES`
- grid threshold/alpha search runs in parallel
- final scoring over `EVAL_MAX_IMAGES` runs in parallel

If Colab RAM or Drive stalls, lower `CELL7_GRID_WORKERS` and `CELL7_SCORE_WORKERS` inside the cell.

In [ ]:
# ===== CELL 7: SIMPLE PARALLEL EVALUATE CAMs + STRICT ZERO REPORT =====
%cd {REPO_DIR}

import glob
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from cam.evaluate import entry_stem, resolve_label_path, map_mask_to_synthia16

N_CLASS = 19
CAM_TYPE = 'attn_highres'

# High-speed knobs. If Colab/Drive stalls, lower these to 8-16.
CELL7_LOAD_WORKERS = 32
CELL7_GRID_WORKERS = 32
CELL7_SCORE_WORKERS = 32
CITYSCAPES_19 = [
    'road', 'sidewalk', 'building', 'wall', 'fence',
    'pole', 'traffic light', 'traffic sign', 'vegetation', 'terrain',
    'sky', 'person', 'rider', 'car', 'truck',
    'bus', 'train', 'motorcycle', 'bicycle',
]

# ---- Metrics ----

def fast_hist(label_true, label_pred, n_class):
    mask = (label_true >= 0) & (label_true < n_class)
    lt = label_true[mask].astype(int)
    lp = label_pred[mask].astype(int)
    lp[(lp < 0) | (lp >= n_class)] = n_class
    hist = np.bincount(
        (n_class + 1) * lt + lp,
        minlength=n_class * (n_class + 1),
    ).reshape(n_class, n_class + 1)
    return hist

def scores_from_hist(hist):
    tp = np.diag(hist[:, :N_CLASS])
    gt_count = hist.sum(axis=1)
    pred_count = hist[:, :N_CLASS].sum(axis=0)
    acc = tp.sum() / max(gt_count.sum(), 1.0)
    acc_cls = np.nanmean(tp / np.maximum(gt_count, 1.0))
    iu = tp / np.maximum(gt_count + pred_count - tp, 1.0)
    valid = gt_count > 0
    mean_iu = np.nanmean(iu[valid])
    freq = gt_count / max(gt_count.sum(), 1.0)
    fwavacc = (freq[freq > 0] * iu[freq > 0]).sum()
    return {
        'Pixel Accuracy': float(acc),
        'Mean Accuracy': float(acc_cls),
        'Mean IoU': float(mean_iu),
        'FW IoU': float(fwavacc),
        'Class IoU': dict(zip(range(N_CLASS), iu)),
    }

def count_image_classes(gt):
    counts = np.zeros(N_CLASS, dtype=np.int64)
    valid = gt[(gt >= 0) & (gt < N_CLASS)]
    if valid.size:
        counts[np.unique(valid).astype(np.int64)] = 1
    return counts

def print_per_class_table(title, score, class_img_count):
    print('\n' + title)
    print(f"  {'id':>2s} {'class':<16s} {'#images':>8s} {'IoU':>8s}")
    print(f"  {'--':>2s} {'-'*16} {'-'*8} {'-'*8}")
    class_iou = score['Class IoU']
    for cid, name in enumerate(CITYSCAPES_19):
        count = int(class_img_count[cid]) if class_img_count is not None else 0
        iou = float(class_iou.get(cid, 0.0))
        print(f"  {cid:>2d} {name:<16s} {count:>8d} {iou:>8.4f}")

# ---- Prediction helpers ----

def normalize_per_class(cams):
    out = cams.astype(np.float32, copy=True)
    for c in range(out.shape[0]):
        c_min, c_max = out[c].min(), out[c].max()
        if c_max - c_min > 1e-8:
            out[c] = (out[c] - c_min) / (c_max - c_min)
        else:
            out[c] = 0.0
    return out

def predict_flat_bg(cams, keys, thres):
    bg = np.full((1, cams.shape[1], cams.shape[2]), thres, dtype=cams.dtype)
    c = np.concatenate([bg, cams], axis=0)
    idx = np.argmax(c, axis=0)
    pred = np.full(idx.shape, 255, dtype=np.uint8)
    fg = idx > 0
    pred[fg] = keys[idx[fg] - 1].astype(np.uint8)
    return pred

def predict_adaptive_bg(cams, keys, thres, alpha):
    max_fg = np.max(cams, axis=0, keepdims=True)
    bg = thres * np.power(np.clip(1.0 - max_fg, 0, 1), alpha).astype(cams.dtype)
    c = np.concatenate([bg, cams], axis=0)
    idx = np.argmax(c, axis=0)
    pred = np.full(idx.shape, 255, dtype=np.uint8)
    fg = idx > 0
    pred[fg] = keys[idx[fg] - 1].astype(np.uint8)
    return pred

def predict_by_method(cams_raw, cams_norm, keys, method, thres=0.0, alpha=1.0):
    if method == 'baseline':
        return predict_flat_bg(cams_raw, keys, thres)
    if method == 'norm':
        return predict_flat_bg(cams_norm, keys, thres)
    if method == 'boost':
        return predict_adaptive_bg(cams_norm, keys, thres, alpha)
    if method == 'no_bg':
        return keys[np.argmax(cams_norm, axis=0)].astype(np.uint8)
    raise ValueError(f'Unknown method: {method}')

# ---- I/O ----

def load_cam_gt(cam_dir, gt_root, entry):
    stem = entry_stem(entry)
    cam_path = Path(cam_dir) / f'{stem}.npy'
    gt_path = resolve_label_path(gt_root, entry)
    if not cam_path.exists() or not Path(gt_path).exists():
        return None
    d = np.load(str(cam_path), allow_pickle=True).item()
    if CAM_TYPE not in d:
        return None
    cams_raw = d[CAM_TYPE].astype(np.float32)
    cams_norm = normalize_per_class(cams_raw)
    keys = d['keys'].astype(np.int64)
    gt = np.asarray(Image.open(gt_path), dtype=np.uint8)
    gt = map_mask_to_synthia16(gt)
    return cams_raw, cams_norm, keys, gt

def load_grid_cache(cam_dir, gt_root, entries):
    cache = []
    errors = []
    with ThreadPoolExecutor(max_workers=CELL7_LOAD_WORKERS) as executor:
        future_to_entry = {
            executor.submit(load_cam_gt, cam_dir, gt_root, e): e
            for e in entries
        }
        for fut in tqdm(as_completed(future_to_entry), total=len(future_to_entry), desc=f'Grid loading {Path(cam_dir).name}'):
            entry = future_to_entry[fut]
            try:
                item = fut.result()
            except Exception as e:
                errors.append((entry, repr(e)))
                print(f'[grid-load-error] {entry}: {type(e).__name__}: {e}')
                continue
            if item is not None:
                cache.append(item)
    if errors:
        print(f'[grid-load-error] {len(errors)} grid load errors. First 3: {errors[:3]}')
    return cache

# ---- Scoring ----

def score_cached(cache, method, thres=0.0, alpha=1.0):
    hist = np.zeros((N_CLASS, N_CLASS + 1), dtype=np.float64)
    for cams_raw, cams_norm, keys, gt in cache:
        pred = predict_by_method(cams_raw, cams_norm, keys, method, thres, alpha)
        hist += fast_hist(gt.flatten(), pred.flatten(), N_CLASS)
    return scores_from_hist(hist)

def parallel_grid_search(cache, method, params, desc):
    def run_one(param):
        thres, alpha = param
        miou = score_cached(cache, method, thres, alpha)['Mean IoU']
        return miou, float(thres), float(alpha)

    best_miou, best_thres, best_alpha = -1.0, 0.0, 1.0
    errors = []
    with ThreadPoolExecutor(max_workers=CELL7_GRID_WORKERS) as executor:
        future_to_param = {executor.submit(run_one, p): p for p in params}
        for fut in tqdm(as_completed(future_to_param), total=len(future_to_param), desc=desc, leave=False):
            param = future_to_param[fut]
            try:
                miou, thres, alpha = fut.result()
            except Exception as e:
                errors.append((param, repr(e)))
                print(f'[grid-error] {desc} param={param}: {type(e).__name__}: {e}')
                continue
            if miou > best_miou:
                best_miou, best_thres, best_alpha = miou, thres, alpha
    if errors:
        print(f'[grid-error] {desc}: {len(errors)} failed params. First 3: {errors[:3]}')
    if best_miou < 0:
        raise RuntimeError(f'All grid params failed for {desc}')
    return best_miou, best_thres, best_alpha

def stream_score_protocols(cam_dir, split_file, gt_root, n_images, protocols):
    with open(split_file, 'r') as f:
        entries = [line.strip() for line in f if line.strip()][:n_images]

    def score_one(entry):
        item = load_cam_gt(cam_dir, gt_root, entry)
        if item is None:
            return None
        cams_raw, cams_norm, keys, gt = item
        img_counts = count_image_classes(gt)
        local = {name: np.zeros((N_CLASS, N_CLASS + 1), dtype=np.float64) for name in protocols}
        for name, cfg in protocols.items():
            pred = predict_by_method(
                cams_raw, cams_norm, keys,
                cfg['method'], cfg.get('thres', 0.0), cfg.get('alpha', 1.0)
            )
            local[name] += fast_hist(gt.flatten(), pred.flatten(), N_CLASS)
        return local, img_counts

    hists = {name: np.zeros((N_CLASS, N_CLASS + 1), dtype=np.float64) for name in protocols}
    class_img_count = np.zeros(N_CLASS, dtype=np.int64)
    n_loaded = 0
    errors = []
    with ThreadPoolExecutor(max_workers=CELL7_SCORE_WORKERS) as executor:
        future_to_entry = {executor.submit(score_one, e): e for e in entries}
        for fut in tqdm(as_completed(future_to_entry), total=len(future_to_entry), desc=f'Full scoring {Path(cam_dir).name}'):
            entry = future_to_entry[fut]
            try:
                result = fut.result()
            except Exception as e:
                errors.append((entry, repr(e)))
                print(f'[score-error] {entry}: {type(e).__name__}: {e}')
                continue
            if result is None:
                continue
            local, img_counts = result
            n_loaded += 1
            class_img_count += img_counts
            for name in protocols:
                hists[name] += local[name]

    if errors:
        print(f'[score-error] {len(errors)} scoring errors. First 3: {errors[:3]}')
    if n_loaded == 0:
        raise RuntimeError(f'No valid CAM/GT pairs scored for {cam_dir}')

    return {name: scores_from_hist(hist) for name, hist in hists.items()}, n_loaded, class_img_count

# ---- Main evaluation ----

def boost_evaluate(cam_dir, split_file, gt_root, n_images, kind=None):
    with open(split_file, 'r') as f:
        entries_full = [line.strip() for line in f if line.strip()][:n_images]

    # Do not pick thresholds from only the first 100 images: that can overfit to
    # a weird class mix and make DAMP look artificially bad. Score a small fixed
    # set directly on the full eval subset instead.
    flat_thresholds = [0.01, 0.03, 0.10, 0.20]
    boost_params = [(0.10, 0.5), (0.20, 0.5), (0.50, 0.5), (0.10, 1.0)]

    def tag(x):
        return str(x).replace('.', 'p')

    protocols = {}
    for t in flat_thresholds:
        protocols[f'baseline_t{tag(t)}'] = {'method': 'baseline', 'thres': float(t), 'alpha': 1.0}
        protocols[f'norm_t{tag(t)}'] = {'method': 'norm', 'thres': float(t), 'alpha': 1.0}
    for t, alpha in boost_params:
        protocols[f'boost_t{tag(t)}_a{tag(alpha)}'] = {'method': 'boost', 'thres': float(t), 'alpha': float(alpha)}
    protocols['no_bg'] = {'method': 'no_bg', 'thres': 0.0, 'alpha': 0.0}

    print(f'  Final scoring images: {len(entries_full)}')
    print(f'  Workers: score={CELL7_SCORE_WORKERS}')
    print(f'  Full fixed protocols: {len(protocols)} ({list(protocols.keys())})')

    if kind == 'zero' and ZERO_STRICT_ENABLED:
        protocols['strict'] = {
            'method': ZERO_STRICT_METHOD,
            'thres': ZERO_STRICT_THRES,
            'alpha': ZERO_STRICT_ALPHA,
        }

    full_scores, n_loaded, class_img_count = stream_score_protocols(cam_dir, split_file, gt_root, n_images, protocols)

    def best_for_method(method):
        names = [name for name, cfg in protocols.items() if name != 'strict' and cfg['method'] == method]
        best_key = max(names, key=lambda name: full_scores[name]['Mean IoU'])
        cfg = protocols[best_key]
        score = full_scores[best_key]
        return {
            'miou': score['Mean IoU'],
            'thres': cfg.get('thres', 0.0),
            'alpha': cfg.get('alpha', 1.0),
            'score_key': best_key,
            'score': score,
        }

    best_baseline = best_for_method('baseline')
    best_norm = best_for_method('norm')
    best_boost = best_for_method('boost')
    best_nobg = best_for_method('no_bg')

    print(f"  Baseline (raw, flat bg): mIoU={best_baseline['miou']:.4f} "
          f"(protocol={best_baseline['score_key']}, thres={best_baseline['thres']:.3f})")
    print(f"  Norm only (flat bg):     mIoU={best_norm['miou']:.4f} "
          f"(protocol={best_norm['score_key']}, thres={best_norm['thres']:.3f})")
    print(f"  Boost (norm+adaptive):   mIoU={best_boost['miou']:.4f} "
          f"(protocol={best_boost['score_key']}, thres={best_boost['thres']:.2f}, alpha={best_boost['alpha']:.1f})")
    print(f"  No-bg (argmax fg):       mIoU={best_nobg['miou']:.4f} "
          f"(protocol={best_nobg['score_key']})")
    if 'strict' in full_scores:
        print(f"  Zero strict report:      mIoU={full_scores['strict']['Mean IoU']:.4f} "
              f"(method={ZERO_STRICT_METHOD}, thres={ZERO_STRICT_THRES}, alpha={ZERO_STRICT_ALPHA})")

    candidates = [
        ('baseline', best_baseline['miou']),
        ('norm', best_norm['miou']),
        ('boost', best_boost['miou']),
        ('no_bg', best_nobg['miou']),
    ]
    best_name, best_val = max(candidates, key=lambda x: x[1])
    print(f">>> BEST: {best_name} = {best_val:.4f} ({n_loaded} scored images)")
    print_per_class_table(
        f"Per-class IoU for {kind or Path(cam_dir).name} / {best_name}",
        {'baseline': best_baseline, 'norm': best_norm, 'boost': best_boost, 'no_bg': best_nobg}[best_name]['score'],
        class_img_count,
    )

    result = {
        'baseline': best_baseline,
        'norm': best_norm,
        'boost': best_boost,
        'no_bg': best_nobg,
        'strict': None,
        'best_method': best_name,
        'n_images': n_loaded,
        'class_image_count': class_img_count,
    }
    if 'strict' in full_scores:
        result['strict'] = {
            'miou': full_scores['strict']['Mean IoU'],
            'method': ZERO_STRICT_METHOD,
            'thres': ZERO_STRICT_THRES,
            'alpha': ZERO_STRICT_ALPHA,
        }
    return result

# ---- Run for each CAM kind ----

all_results = {}
for kind, cam_dir in CAM_DIR_BY_KIND.items():
    n_cam = len(glob.glob(str(cam_dir / '*.npy')))
    print(' ' + '=' * 80)
    print(f'Evaluating {kind}: {cam_dir} ({n_cam} CAM files)')
    if n_cam == 0:
        print('  SKIP: no CAM files')
        continue
    all_results[kind] = boost_evaluate(
        str(cam_dir), str(SYNTHIA_CAM_SPLIT), str(SYNTHIA_LBL), EVAL_MAX_IMAGES, kind=kind
    )

# ---- Summary ----

print(' ' + '=' * 80)
print('SUMMARY: exact mIoU on EVAL_MAX_IMAGES with fixed full-eval protocol selection')
print('=' * 80)
print(f"  {'Kind':<14s} {'Use':>5s} {'Baseline':>10s} {'Norm':>10s} {'Boost':>10s} {'No-bg':>10s} {'Strict':>10s}  Best")
print(f"  {'-'*14} {'-'*5} {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*10}  {'-'*10}")
for kind in CAM_DIR_BY_KIND:
    r = all_results.get(kind, {})
    if not r:
        continue
    strict = r.get('strict')
    strict_str = f"{strict['miou']:.4f}" if strict else '-'
    use_tag = 'pick' if kind in SELECTABLE_CAM_KINDS else 'report'
    print(f"  {kind:<14s} {use_tag:>5s} {r['baseline']['miou']:>10.4f} {r['norm']['miou']:>10.4f} "
          f"{r['boost']['miou']:>10.4f} {r['no_bg']['miou']:>10.4f} {strict_str:>10s}  {r['best_method']}")

# ---- Auto-pick best DAMP-family kind + method. Zero-shot is report-only. ----

def result_best_miou(r):
    return max(r['baseline']['miou'], r['norm']['miou'], r['boost']['miou'], r['no_bg']['miou'])

selectable_results = {k: all_results[k] for k in SELECTABLE_CAM_KINDS if k in all_results and all_results[k]}
if not selectable_results:
    raise RuntimeError(f'No selectable DAMP CAM results found. Expected one of {SELECTABLE_CAM_KINDS}.')
print(f"Selectable CAM kinds for pseudo-mask generation: {list(selectable_results.keys())}")
best_kind = max(selectable_results.keys(), key=lambda k: result_best_miou(selectable_results[k]))
r = all_results[best_kind]
BEST_CAM_KIND = best_kind
BEST_CAM_DIR = CAM_DIR_BY_KIND[BEST_CAM_KIND]
BEST_METHOD = r['best_method']

if BEST_METHOD == 'boost':
    BEST_THRES = r['boost']['thres']
    BEST_ALPHA = r['boost']['alpha']
elif BEST_METHOD == 'norm':
    BEST_THRES = r['norm']['thres']
    BEST_ALPHA = 1.0
elif BEST_METHOD == 'baseline':
    BEST_THRES = r['baseline']['thres']
    BEST_ALPHA = 1.0
else:  # no_bg
    BEST_THRES = 0.0
    BEST_ALPHA = 0.0

print(f">>> BEST CAM KIND : {BEST_CAM_KIND}")
print(f"  >>> BEST METHOD   : {BEST_METHOD}")
print(f"  >>> BEST THRES    : {BEST_THRES}")
print(f"  >>> BEST ALPHA    : {BEST_ALPHA}")
if 'class_image_count' in r:
    print_per_class_table(
        f"FINAL SELECTED PER-CLASS: {BEST_CAM_KIND} / {BEST_METHOD}",
        r[BEST_METHOD]['score'],
        r['class_image_count'],
    )
print('Cell 8 will use these params to generate pseudo masks.')

## Step 8: Create Pseudo Masks for Zero-shot, DAMP Prompt-only, and DAMP Full

Generate pseudo-mask PNGs for all three report methods: `zero`, `prompt_only`, and `damp_full`.

Each method uses its own best post-processing parameters from Cell 7. Output folders include method, threshold, and alpha so reruns do not reuse stale masks.

In [ ]:
# ===== CELL 8: ZERO / DAMP PROMPT-ONLY / DAMP FULL CAMs -> PSEUDO-MASKS =====
%cd {REPO_DIR}

import glob
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from cam.evaluate import entry_stem

PSEUDO_MASK_KINDS = ('zero', 'prompt_only', 'damp_full')
SEG_EXPORT_DIR = OUTPUT_DIR / 'segmentation' / RUN_NAME

def _params_for_kind(kind):
    if kind not in all_results or not all_results[kind]:
        raise RuntimeError(f'Missing Cell 7 evaluation result for {kind}. Run Cell 7 first.')
    r = all_results[kind]
    method = r['best_method']
    if method == 'boost':
        return method, r['boost']['thres'], r['boost']['alpha']
    if method == 'norm':
        return method, r['norm']['thres'], 1.0
    if method == 'baseline':
        return method, r['baseline']['thres'], 1.0
    if method == 'no_bg':
        return method, 0.0, 0.0
    raise ValueError(f'Unknown best method for {kind}: {method}')

def _mask_dir_for(kind, method, thres, alpha):
    method_safe = str(method).replace('-', '_')
    thres_tag = f"t{int(round(float(thres) * 1000)):03d}"
    alpha_tag = f"a{str(alpha).replace('.', 'p')}"
    return OUTPUT_DIR / 'synthia' / f'pseudo_masks_{kind}_{method_safe}_{thres_tag}_{alpha_tag}_{RUN_NAME}_{CAM_MAX_IMAGES}'

def _normalize_per_class(cams):
    out = cams.astype(np.float32, copy=True)
    for c in range(out.shape[0]):
        c_min, c_max = out[c].min(), out[c].max()
        if c_max - c_min > 1e-8:
            out[c] = (out[c] - c_min) / (c_max - c_min)
        else:
            out[c] = 0.0
    return out

def _predict_mask(cams_raw, keys, method, thres, alpha):
    cams = _normalize_per_class(cams_raw) if method in ('boost', 'norm', 'no_bg') else cams_raw.astype(np.float32)
    if method == 'no_bg':
        return keys[np.argmax(cams, axis=0)].astype(np.uint8)
    if method == 'boost':
        max_fg = np.max(cams, axis=0, keepdims=True)
        bg = thres * np.power(np.clip(1.0 - max_fg, 0, 1), alpha).astype(cams.dtype)
    else:
        bg = np.full((1, cams.shape[1], cams.shape[2]), thres, dtype=cams.dtype)
    merged = np.concatenate([bg, cams], axis=0)
    idx = np.argmax(merged, axis=0)
    pred = np.full(idx.shape, 255, dtype=np.uint8)
    fg = idx > 0
    pred[fg] = keys[idx[fg] - 1].astype(np.uint8)
    return pred

with open(SYNTHIA_CAM_SPLIT, 'r') as f:
    entries = [line.strip() for line in f if line.strip()][:CAM_MAX_IMAGES]

PSEUDO_MASK_RUNS = {}

for kind in PSEUDO_MASK_KINDS:
    cam_dir = CAM_DIR_BY_KIND[kind]
    method, thres, alpha = _params_for_kind(kind)
    mask_dir = _mask_dir_for(kind, method, thres, alpha)
    mask_dir.mkdir(parents=True, exist_ok=True)
    print('\n' + '=' * 80)
    print(f'Pseudo masks for {kind}:')
    print(f'  cam_dir : {cam_dir}')
    print(f'  method  : {method}, thres={thres}, alpha={alpha}')
    print(f'  mask_dir: {mask_dir}')

    n_existing = len(glob.glob(str(mask_dir / '*.png')))
    if n_existing >= CAM_MAX_IMAGES:
        print(f'  Pseudo masks already exist ({n_existing}). Skipping generation.')
    else:
        saved = 0
        missing = []
        for entry in tqdm(entries, desc=f'pseudo masks {kind}'):
            stem = entry_stem(entry)
            cam_path = cam_dir / f'{stem}.npy'
            out_path = mask_dir / f'{stem}.png'
            if out_path.exists():
                saved += 1
                continue
            if not cam_path.exists():
                missing.append(stem)
                continue
            d = np.load(str(cam_path), allow_pickle=True).item()
            cams = d['attn_highres'].astype(np.float32)
            keys = d['keys'].astype(np.int64)
            pred = _predict_mask(cams, keys, method, thres, alpha)
            Image.fromarray(pred.astype(np.uint8)).save(out_path)
            saved += 1
        if missing:
            print(f'  WARNING: missing {len(missing)} CAM files. First 10: {missing[:10]}')
        print(f'  Saved/kept {saved} pseudo masks.')

    n_masks = len(glob.glob(str(mask_dir / '*.png')))
    if n_masks == 0:
        raise RuntimeError(f'No pseudo masks generated for {kind}: {mask_dir}')
    PSEUDO_MASK_RUNS[kind] = {
        'cam_dir': cam_dir,
        'mask_dir': mask_dir,
        'method': method,
        'thres': float(thres),
        'alpha': float(alpha),
        'n_masks': int(n_masks),
    }

print('\nPseudo-mask runs ready:')
for kind, run in PSEUDO_MASK_RUNS.items():
    print(f"  {kind:<12s} masks={run['n_masks']:>5d} method={run['method']} dir={run['mask_dir']}")

# Default export target until Cell 8b selects by Stage 3 mIoU.
EXPORT_MASK_KIND = 'damp_full' if 'damp_full' in PSEUDO_MASK_RUNS else next(iter(PSEUDO_MASK_RUNS))
MASK_DIR = PSEUDO_MASK_RUNS[EXPORT_MASK_KIND]['mask_dir']
method_safe = str(PSEUDO_MASK_RUNS[EXPORT_MASK_KIND]['method']).replace('-', '_')
SEG_TRAIN_PAIRS = SEG_EXPORT_DIR / f'train_pairs_{EXPORT_MASK_KIND}_{method_safe}_first{CAM_MAX_IMAGES}.txt'
print(f'\nDefault Step 9 export mask kind: {EXPORT_MASK_KIND}')
print(f'MASK_DIR       : {MASK_DIR}')
print(f'SEG_TRAIN_PAIRS: {SEG_TRAIN_PAIRS}')

## Step 8b: Evaluate Stage 3 Pseudo-mask mIoU

Evaluate all pseudo-mask PNG sets from Step 8 against SYNTHIA labels. These are the Stage 3 pseudo-mask mIoU numbers for the report.

In [ ]:
# ===== CELL 8b: EVALUATE STAGE 3 PSEUDO-MASKS =====
%cd {REPO_DIR}

import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from cam.evaluate import entry_stem, resolve_label_path, map_mask_to_synthia16
from cam.clip_text import CITYSCAPES_CLASS_NAMES

N_CLASS = 19

def _stage3_fast_hist(label_true, label_pred, n_class):
    mask = (label_true >= 0) & (label_true < n_class)
    lt = label_true[mask].astype(int)
    lp = label_pred[mask].astype(int)
    lp[(lp < 0) | (lp >= n_class)] = n_class
    return np.bincount(
        (n_class + 1) * lt + lp,
        minlength=n_class * (n_class + 1),
    ).reshape(n_class, n_class + 1)

def _stage3_scores(hist):
    tp = np.diag(hist[:, :N_CLASS])
    gt_count = hist.sum(axis=1)
    pred_count = hist[:, :N_CLASS].sum(axis=0)
    acc = tp.sum() / max(gt_count.sum(), 1.0)
    mean_acc = np.nanmean(tp / np.maximum(gt_count, 1.0))
    iu = tp / np.maximum(gt_count + pred_count - tp, 1.0)
    valid = gt_count > 0
    mean_iu = np.nanmean(iu[valid])
    freq = gt_count / max(gt_count.sum(), 1.0)
    fw_iou = (freq[freq > 0] * iu[freq > 0]).sum()
    return acc, mean_acc, mean_iu, fw_iou, iu, gt_count

with open(SYNTHIA_CAM_SPLIT, 'r') as f:
    entries = [line.strip() for line in f if line.strip()][:CAM_MAX_IMAGES]

if 'PSEUDO_MASK_RUNS' not in globals() or not PSEUDO_MASK_RUNS:
    raise RuntimeError('PSEUDO_MASK_RUNS is missing. Run Cell 8 first.')

def _evaluate_stage3_mask_dir(kind, run):
    mask_dir = run['mask_dir']
    hist = np.zeros((N_CLASS, N_CLASS + 1), dtype=np.float64)
    class_image_count = np.zeros(N_CLASS, dtype=np.int64)
    loaded = 0
    missing = []

    for entry in tqdm(entries, desc=f'Stage 3 eval {kind}'):
        stem = entry_stem(entry)
        pred_path = mask_dir / f'{stem}.png'
        gt_path = resolve_label_path(str(SYNTHIA_LBL), entry)
        if not pred_path.exists() or not Path(gt_path).exists():
            missing.append(stem)
            continue
        pred = np.asarray(Image.open(pred_path), dtype=np.uint8)
        gt = np.asarray(Image.open(gt_path), dtype=np.uint8)
        gt = map_mask_to_synthia16(gt)
        for c in np.unique(gt):
            c = int(c)
            if 0 <= c < N_CLASS:
                class_image_count[c] += 1
        hist += _stage3_fast_hist(gt.flatten(), pred.flatten(), N_CLASS)
        loaded += 1

    if missing:
        print(f'WARNING [{kind}]: missing {len(missing)} pseudo masks/GTs. First 10: {missing[:10]}')
    if loaded == 0:
        raise RuntimeError(f'No pseudo masks loaded for {kind}: {mask_dir}')

    pa, ma, miou, fwiou, class_iou, gt_count = _stage3_scores(hist)
    score = {
        'Pixel Accuracy': float(pa),
        'Mean Accuracy': float(ma),
        'Mean IoU': float(miou),
        'FW IoU': float(fwiou),
        'Class IoU': {i: float(class_iou[i]) for i in range(N_CLASS)},
        'Loaded': int(loaded),
        'Method': run['method'],
        'Threshold': float(run['thres']),
        'Alpha': float(run['alpha']),
        'Mask Dir': str(mask_dir),
    }

    print('\n' + '=' * 80)
    print(f'Stage 3 pseudo-mask evaluation: {kind}')
    print(f'  masks loaded    : {loaded}/{len(entries)}')
    print(f'  method          : {run["method"]}, thres={run["thres"]}, alpha={run["alpha"]}')
    print(f'  mask dir        : {mask_dir}')
    print(f'  Pixel Accuracy  : {pa:.4f}')
    print(f'  Mean Accuracy   : {ma:.4f}')
    print(f'  Mean IoU        : {miou:.4f}')
    print(f'  FW IoU          : {fwiou:.4f}')

    print('\nPer-class IoU')
    print(f"  {'id':>2s} {'class':<16s} {'#images':>8s} {'IoU':>8s}")
    print(f"  {'--':>2s} {'-'*16:<16s} {'-'*8:>8s} {'-'*8:>8s}")
    for cid, cname in enumerate(CITYSCAPES_CLASS_NAMES):
        print(f"  {cid:>2d} {cname:<16s} {int(class_image_count[cid]):>8d} {class_iou[cid]:>8.4f}")
    return score

STAGE3_PSEUDO_MASK_SCORES = {}
for kind, run in PSEUDO_MASK_RUNS.items():
    STAGE3_PSEUDO_MASK_SCORES[kind] = _evaluate_stage3_mask_dir(kind, run)

print('\n' + '=' * 80)
print('STAGE 3 SUMMARY: pseudo-mask mIoU')
print(f"  {'kind':<12s} {'method':<10s} {'mIoU':>8s} {'PA':>8s} {'MA':>8s} {'FWIoU':>8s} {'loaded':>8s}")
print(f"  {'-'*12:<12s} {'-'*10:<10s} {'-'*8:>8s} {'-'*8:>8s} {'-'*8:>8s} {'-'*8:>8s} {'-'*8:>8s}")
for kind, score in STAGE3_PSEUDO_MASK_SCORES.items():
    print(f"  {kind:<12s} {score['Method']:<10s} {score['Mean IoU']:>8.4f} {score['Pixel Accuracy']:>8.4f} {score['Mean Accuracy']:>8.4f} {score['FW IoU']:>8.4f} {score['Loaded']:>8d}")

STAGE3_BEST_KIND = max(STAGE3_PSEUDO_MASK_SCORES, key=lambda k: STAGE3_PSEUDO_MASK_SCORES[k]['Mean IoU'])
EXPORT_MASK_KIND = STAGE3_BEST_KIND
MASK_DIR = PSEUDO_MASK_RUNS[EXPORT_MASK_KIND]['mask_dir']
method_safe = str(PSEUDO_MASK_RUNS[EXPORT_MASK_KIND]['method']).replace('-', '_')
SEG_TRAIN_PAIRS = SEG_EXPORT_DIR / f'train_pairs_{EXPORT_MASK_KIND}_{method_safe}_first{CAM_MAX_IMAGES}.txt'
print(f'\nBest Stage 3 pseudo-mask kind: {STAGE3_BEST_KIND}')
print(f'Cell 9 will export pairs from: {MASK_DIR}')


## Step 8c: Exp 3 Qualitative Hybrid Panels

Export visual panels for the report: original image, zero-shot pseudo-label, DAMP prompt-only pseudo-label, DAMP full pseudo-label, hybrid pseudo-label, and GT/reference.

In [ ]:
# ===== CELL 8c: EXP 3 QUALITATIVE PANELS =====
%cd {REPO_DIR}

import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display
from cam.evaluate import entry_stem, resolve_label_path, map_mask_to_synthia16
from cam.clip_text import CITYSCAPES_CLASS_NAMES

QUAL_N_EXAMPLES = 3
QUAL_OUTPUT_DIR = OUTPUT_DIR / 'figures' / RUN_NAME
QUAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CITYSCAPES_COLORS = np.array([
    [128, 64,128], [244, 35,232], [ 70, 70, 70], [102,102,156], [190,153,153],
    [153,153,153], [250,170, 30], [220,220,  0], [107,142, 35], [152,251,152],
    [ 70,130,180], [220, 20, 60], [255,  0,  0], [  0,  0,142], [  0,  0, 70],
    [  0, 60,100], [  0, 80,100], [  0,  0,230], [119, 11, 32],
], dtype=np.uint8)

def _image_path_for(entry):
    stem = entry_stem(entry)
    candidates = [SYNTHIA_IMG / Path(entry).name, SYNTHIA_IMG / f'{stem}.png', SYNTHIA_IMG / f'{stem}.jpg']
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Image not found for {entry}')

def _colorize_mask(mask):
    out = np.zeros((*mask.shape, 3), dtype=np.uint8)
    valid = (mask >= 0) & (mask < 19)
    out[valid] = CITYSCAPES_COLORS[mask[valid].astype(np.int64)]
    out[mask == 255] = np.array([30, 30, 30], dtype=np.uint8)
    return out

def _overlay_mask(image, mask, alpha=0.55):
    color = _colorize_mask(mask).astype(np.float32)
    base = image.astype(np.float32)
    valid = mask != 255
    out = base.copy()
    out[valid] = (1 - alpha) * base[valid] + alpha * color[valid]
    return np.clip(out, 0, 255).astype(np.uint8)

def _predict_from_cam_for_kind(entry, kind):
    stem = entry_stem(entry)
    if kind in globals().get('PSEUDO_MASK_RUNS', {}):
        mask_path = PSEUDO_MASK_RUNS[kind]['mask_dir'] / f'{stem}.png'
        if mask_path.exists():
            return np.asarray(Image.open(mask_path), dtype=np.uint8)
    method, thres, alpha = _params_for_kind(kind)
    cam_path = CAM_DIR_BY_KIND[kind] / f'{stem}.npy'
    if not cam_path.exists():
        raise FileNotFoundError(f'CAM missing for {kind}: {cam_path}')
    d = np.load(str(cam_path), allow_pickle=True).item()
    return _predict_mask(d['attn_highres'].astype(np.float32), d['keys'].astype(np.int64), method, thres, alpha)

with open(SYNTHIA_CAM_SPLIT, 'r') as f:
    qual_entries_all = [line.strip() for line in f if line.strip()][:CAM_MAX_IMAGES]

required_kinds = ['zero', 'prompt_only', 'damp_full', 'hybrid']
qual_entries = []
for entry in qual_entries_all:
    stem = entry_stem(entry)
    if all((CAM_DIR_BY_KIND[k] / f'{stem}.npy').exists() for k in required_kinds):
        gt_path = Path(resolve_label_path(str(SYNTHIA_LBL), entry))
        if gt_path.exists():
            gt = map_mask_to_synthia16(np.asarray(Image.open(gt_path), dtype=np.uint8))
            if len([c for c in np.unique(gt) if 0 <= int(c) < 19]) >= 4:
                qual_entries.append(entry)
    if len(qual_entries) >= QUAL_N_EXAMPLES:
        break

if len(qual_entries) < QUAL_N_EXAMPLES:
    raise RuntimeError(f'Only found {len(qual_entries)} qualitative examples with all required CAMs.')

columns = [
    ('Image', None),
    ('Zero-shot', 'zero'),
    ('DAMP prompt-only', 'prompt_only'),
    ('DAMP full', 'damp_full'),
    ('Hybrid', 'hybrid'),
    ('GT / reference', 'gt'),
]

fig, axes = plt.subplots(len(qual_entries), len(columns), figsize=(4 * len(columns), 3.2 * len(qual_entries)))
if len(qual_entries) == 1:
    axes = axes[None, :]

for row, entry in enumerate(qual_entries):
    image = np.asarray(Image.open(_image_path_for(entry)).convert('RGB'))
    gt_path = resolve_label_path(str(SYNTHIA_LBL), entry)
    gt = map_mask_to_synthia16(np.asarray(Image.open(gt_path), dtype=np.uint8))
    for col, (title, kind) in enumerate(columns):
        ax = axes[row, col]
        if kind is None:
            vis = image
        elif kind == 'gt':
            vis = _overlay_mask(image, gt, alpha=0.55)
        else:
            pred = _predict_from_cam_for_kind(entry, kind)
            if pred.shape[:2] != image.shape[:2]:
                pred = np.asarray(Image.fromarray(pred).resize((image.shape[1], image.shape[0]), Image.NEAREST), dtype=np.uint8)
            vis = _overlay_mask(image, pred, alpha=0.55)
        ax.imshow(vis)
        ax.axis('off')
        if row == 0:
            ax.set_title(title, fontsize=12)
        if col == 0:
            ax.set_ylabel(f'Example {row + 1}\n{entry_stem(entry)}', fontsize=11)

fig.tight_layout()
panel_path = QUAL_OUTPUT_DIR / f'exp3_hybrid_qualitative_{RUN_NAME}_{CAM_MAX_IMAGES}.png'
fig.savefig(panel_path, dpi=180, bbox_inches='tight')
plt.show()

print(f'Saved qualitative panel: {panel_path}')
print('Selected examples:')
for entry in qual_entries:
    print(' ', entry)


## Step 9: Export Segmentation Training Pairs

This writes `image_path mask_path` pairs for the selected pseudo-mask set.


In [ ]:
# ===== CELL 9: EXPORT IMAGE/MASK PAIRS FOR SEGMENTATION TRAINING =====
from pathlib import Path

SEG_EXPORT_DIR = OUTPUT_DIR / 'segmentation' / RUN_NAME
SEG_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

with open(SYNTHIA_CAM_SPLIT, 'r') as f:
    entries = [line.strip() for line in f if line.strip()]

pairs = []
missing = []
for entry in entries:
    stem = Path(entry).stem
    image_path = SYNTHIA_IMG / f'{stem}.png'
    if not image_path.exists():
        image_path = SYNTHIA_IMG / Path(entry).name
    mask_path = MASK_DIR / f'{stem}.png'

    if image_path.exists() and mask_path.exists():
        pairs.append((image_path, mask_path))
    else:
        missing.append((image_path, mask_path))

with open(SEG_TRAIN_PAIRS, 'w') as f:
    for image_path, mask_path in pairs:
        f.write(f'{image_path} {mask_path}\n')

print(f'Exported {len(pairs)} image/mask pairs -> {SEG_TRAIN_PAIRS}')
if missing:
    print(f'WARNING: {len(missing)} entries missing image or mask. First 5:')
    for image_path, mask_path in missing[:5]:
        print(' ', image_path, '|', mask_path)

print('\nSegmentation training inputs:')
print(f'  image_dir : {SYNTHIA_IMG}')
print(f'  mask_dir  : {MASK_DIR}')
print(f'  pair_file : {SEG_TRAIN_PAIRS}')
print(f'  ignore_id : 255')
print(f'  classes   : Cityscapes train IDs, SYNTHIA-valid subset')
